# Block 5 — Lateral electromagnetic shower distribution

This notebook validates the deterministic mean lateral model used by the AMS-02 ECAL FastMC. It connects three pieces of physics and detector geometry:

1. the Molière radius sets the natural transverse scale of an electromagnetic shower;
2. the AMS test-beam parametrization describes how the lateral scale changes with layer and primary energy;
3. alternating fiber layers observe a one-dimensional projection of the radial shower around the tracker-projected axis.

Stochastic shower-to-shower fluctuations are intentionally excluded until Block 6.

## Physics model

The normalized energy density in one transverse plane is

$$
\rho(r) = \frac{3R^2}{\pi(r+R)^4},
$$

where $r$ is distance from the shower axis and $R$ is a phenomenological scale. Integrating $\rho$ over the whole plane gives one.

The AMS test-beam fit makes the scale depend on layer number $l$ and primary energy $E$. This repository maps $l$ to its established zero-based layer index:

$$
R_l = A(E)l^2 + B, \qquad A(E)=p_0\ln(E/1\,\mathrm{GeV})+p_1.
$$

The fitted $R_l$ is expressed in calibration-cell units. One calibration cell is approximately half a Molière radius. For the ideal 9 mm ECAL pitch, the geometry therefore supplies a nominal Molière radius of 18 mm.

In [ ]:
from math import pi, radians
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad

from ams_ecal import (
    AMSLateralShowerModel,
    TrackState,
    load_fastmc_config,
    load_geometry,
)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

geometry = load_geometry(PROJECT_ROOT / "configs" / "geometry.yaml")
fastmc_config = load_fastmc_config(PROJECT_ROOT / "configs" / "fastmc.yaml")
model = AMSLateralShowerModel(fastmc_config.lateral_em, geometry)

print(f"Nominal Moliere radius: {model.nominal_moliere_radius_mm:.1f} mm")
print(f"Cell pitch: {geometry.cell_pitch_mm:.1f} mm")
print(f"Calibration-cell fraction: {model.config.calibration_cell_moliere_fraction:.1f} R_M")

## Layer- and energy-dependent scale

The fit predicts a narrow entrance profile that broadens with shower depth. Its published calibration used 3–180 GeV electrons. Curves above 180 GeV are explicit extrapolations, not new calibration claims.

In [ ]:
energies_mev = (3_000.0, 10_000.0, 100_000.0, 180_000.0, 1_000_000.0)
layer_indices = np.arange(geometry.number_of_layers)

fig, ax = plt.subplots(figsize=(9, 5))
for energy_mev in energies_mev:
    scales_mm = [
        model.layer_scale_mm(int(layer), energy_mev)
        for layer in layer_indices
    ]
    status = "fit range" if model.is_within_calibration_range(energy_mev) else "extrapolation"
    ax.plot(layer_indices, scales_mm, marker="o", label=f"{energy_mev / 1000:g} GeV ({status})")

ax.set(
    xlabel="Zero-based ECAL layer index",
    ylabel="Lateral scale R [mm]",
    title="AMS mean lateral scale versus shower depth",
)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Normalization and radial containment

A valid energy-density model must conserve the layer energy before detector boundaries are applied:

$$
\int_0^\infty 2\pi r\rho(r)\,dr = 1.
$$

In [ ]:
energy_mev = 100_000.0
layer_index = 10
normalization, integration_error = quad(
    lambda radius_mm: 2 * pi * radius_mm * model.radial_energy_density_per_mm2(
        radius_mm, layer_index, energy_mev
    ),
    0.0,
    np.inf,
)

radii_mm = np.linspace(0.0, 4.0 * model.nominal_moliere_radius_mm, 400)
containment = [
    model.radial_contained_fraction(float(radius), layer_index, energy_mev)
    for radius in radii_mm
]

print(f"Plane normalization: {normalization:.12f} ± {integration_error:.2e}")
print(
    "Containment within nominal R_M: "
    f"{model.radial_contained_fraction(model.nominal_moliere_radius_mm, layer_index, energy_mev):.4f}"
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(radii_mm / model.nominal_moliere_radius_mm, containment)
ax.axvline(1.0, color="black", linestyle="--", label="nominal $R_M$")
ax.set(xlabel="Radius / $R_M$", ylabel="Contained fraction", title="Radial containment in layer 10 at 100 GeV")
ax.set_ylim(0.0, 1.02)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## From a radial shower to 72 readout cells

Fibers integrate light along their own direction, so a layer observes the one-dimensional marginal of $\rho(r)$ along the coordinate perpendicular to the fibers. The model integrates that marginal over every finite 9 mm cell. It does **not** renormalize the 72 values: missing probability represents lateral leakage beyond the measured transverse boundary.

In [ ]:
cell_centers_mm = (
    -geometry.width_x_mm / 2
    + (np.arange(geometry.cells_per_layer) + 0.5) * geometry.cell_pitch_mm
)
central = np.asarray(model.cell_energy_fractions(0.0, 17, energy_mev))
near_edge = np.asarray(model.cell_energy_fractions(300.0, 17, energy_mev))

print(f"Central containment: {central.sum():.6f}")
print(f"Near-edge containment: {near_edge.sum():.6f}")
print(f"Near-edge lateral leakage: {1.0 - near_edge.sum():.6f}")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(cell_centers_mm, central, marker=".", label="axis at 0 mm")
ax.plot(cell_centers_mm, near_edge, marker=".", label="axis at 300 mm")
ax.set(xlabel="Measured cell coordinate [mm]", ylabel="Mean layer-energy fraction", title="Finite-cell integration and lateral leakage")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Alternating 18 × 72 track-centered representation

For each layer, the tracker state is projected to the layer center. X-directed fibers measure the y coordinate; y-directed fibers measure x. The changing horizontal position below is therefore a combination of track inclination, alternating views, and depth-dependent shower width.

In [ ]:
track = TrackState(
    x0_mm=-18.0,
    y0_mm=12.0,
    z0_mm=0.0,
    theta_rad=radians(8.0),
    phi_rad=radians(35.0),
)
lateral_grid = np.asarray(
    model.track_centered_cell_fractions(track, energy_mev)
)

fig, ax = plt.subplots(figsize=(12, 5))
image = ax.imshow(lateral_grid, aspect="auto", origin="upper", cmap="magma")
ax.set(xlabel="Transverse cell index", ylabel="Longitudinal layer index", title="Block-5 deterministic lateral fractions")
fig.colorbar(image, ax=ax, label="Fraction of that layer's mean energy")
plt.show()

## Scientific invariants

These checks are part of the model contract rather than visual preferences.

In [ ]:
assert abs(normalization - 1.0) < 1e-10
assert lateral_grid.shape == (18, 72)
assert np.all(np.isfinite(lateral_grid))
assert np.all(lateral_grid >= 0.0)
assert np.all(lateral_grid.sum(axis=1) <= 1.0 + 1e-12)
assert central.sum() < 1.0
assert near_edge.sum() < central.sum()
assert np.allclose(central, central[::-1])
print("All Block-5 lateral invariants passed.")

## Scope and handoff to Block 6

Block 5 now provides a deterministic mean lateral backbone. It deliberately does not claim a complete flight-like shower. Important limitations are explicit:

- the quoted fit was calibrated with 3–180 GeV electron test-beam data; higher energies are extrapolations;
- electron and positron mean profiles are treated identically;
- the cell marginal assumes an effectively infinite fiber direction, so finite fiber-end leakage is not yet modeled;
- sampling fluctuations, correlations, shower-start fluctuations, and event-to-event changes in $R_l$ belong to Block 6;
- detector response, attenuation, saturation, thresholds, and dead channels remain Block 7.

The next block can combine the validated longitudinal layer energies with these cell fractions and then introduce reproducible correlated fluctuations.